# Week 7 - Activity 1: Retrieval-Augmented Generation (RAG)

In this activity, we'll explore RAG using different retrieval methods:
1. BM25 (sparse retrieval)
2. Dense embeddings
3. Testing robustness to question paraphrasing

We'll use a small knowledge base and analyze how different retrieval methods affect the quality of generated answers.

In [1]:
import numpy as np
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel
import torch
import faiss
from tqdm.notebook import tqdm

## 1. Create a Small Knowledge Base

Let's create a small knowledge base about a specific topic:

In [ ]:
# Sample knowledge base about machine learning concepts
knowledge_base = [
    {
        'title': 'Gradient Descent',
        'content': '''
        Gradient descent is an optimization algorithm used to minimize a function by iteratively moving in the direction
        of steepest descent. In machine learning, it's commonly used to minimize the loss function.
        The algorithm works by:
        1. Computing the gradient (derivative) of the loss function
        2. Taking a step in the opposite direction of the gradient
        3. Repeating until convergence
        The learning rate determines the size of these steps.
        '''
    },
    {
        'title': 'Backpropagation',
        'content': '''
        Backpropagation is an algorithm for training neural networks that calculates gradients using the chain rule.
        It works by propagating errors backward through the network, from the output layer to the input layer.
        Key steps include:
        1. Forward pass to compute predictions
        2. Computing the loss
        3. Computing gradients using the chain rule
        4. Updating weights
        '''
    },
    {
        'title': 'Overfitting',
        'content': '''
        Overfitting occurs when a model learns the training data too well, including noise and outliers.
        This results in poor generalization to new data. Common solutions include:
        1. Using regularization
        2. Collecting more training data
        3. Reducing model complexity
        4. Early stopping
        5. Cross-validation
        '''
    },
    {
        'title': 'Cross-Validation',
        'content': '''
        Cross-validation is a technique to assess model performance by splitting data into multiple folds.
        In k-fold cross-validation:
        1. Data is split into k folds
        2. Model is trained on k-1 folds
        3. Tested on the remaining fold
        4. Process repeated k times
        This helps estimate how well the model generalizes.
        '''
    }
]

# Prepare documents for retrieval
documents = [f"{doc['title']}\n{doc['content']}" for doc in knowledge_base]
print(f"Knowledge base size: {len(documents)} documents")

## 2. BM25 Retrieval

First, let's implement BM25 retrieval:

In [ ]:
def tokenize(text):
    return text.lower().split()

# Initialize BM25
tokenized_docs = [tokenize(doc) for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, k=2):
    tokenized_query = tokenize(query)
    doc_scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[-k:]
    return [(documents[i], doc_scores[i]) for i in top_indices]

# Test BM25
test_query = "How does gradient descent work?"
print("BM25 Results:")
for doc, score in bm25_search(test_query):
    print(f"\nScore: {score:.3f}")
    print(f"Document: {doc[:200]}...")

## 3. Dense Retrieval

Now let's implement dense retrieval using sentence transformers:

In [ ]:
# Initialize transformer model and tokenizer
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def get_embeddings(texts):
    # Tokenize sentences
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
    
    # Compute token embeddings
    with torch.no_grad():
        model_output = model(**encoded_input)
    
    # Perform pooling
    embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    
    # Normalize embeddings
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    return embeddings.numpy()

# Create document embeddings
doc_embeddings = get_embeddings(documents)

# Create FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

def dense_search(query, k=2):
    query_embedding = get_embeddings([query])
    distances, indices = index.search(query_embedding, k)
    return [(documents[i], float(distances[0][j])) for j, i in enumerate(indices[0])]

# Test dense retrieval
print("\nDense Retrieval Results:")
for doc, distance in dense_search(test_query):
    print(f"\nDistance: {distance:.3f}")
    print(f"Document: {doc[:200]}...")

## 4. RAG with LLM

Let's combine retrieval with an LLM to generate answers:

In [ ]:
import os
from litellm import completion

def generate_answer(query, retrieved_docs):
    context = "\n\n".join([doc for doc, _ in retrieved_docs])
    
    prompt = f"""Based on the following context, answer the question. Use only the information provided in the context.

Context:
{context}

Question: {query}

Answer:"""

    response = completion(
        model=os.getenv('LLM_MODEL'),
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=150,
        api_base=os.getenv('LLM_BASE_URL'),
        api_key=os.getenv('LLM_API_KEY')
    )
    
    return response.choices[0].message.content

# Test questions with different retrieval methods
test_questions = [
    "How does gradient descent minimize the loss function?",
    "What are the steps in the backpropagation algorithm?",
    "How can we prevent overfitting in machine learning models?"
]

print("Comparing BM25 and Dense Retrieval:")
for question in test_questions:
    print(f"\nQuestion: {question}")
    
    print("\nBM25 + LLM:")
    bm25_docs = bm25_search(question)
    bm25_answer = generate_answer(question, bm25_docs)
    print(bm25_answer)
    
    print("\nDense Retrieval + LLM:")
    dense_docs = dense_search(question)
    dense_answer = generate_answer(question, dense_docs)
    print(dense_answer)

## 5. Testing Robustness to Paraphrasing

Let's see how well the retrieval methods handle different ways of asking the same question:

In [ ]:
paraphrase_examples = [
    {
        'original': "How does gradient descent work?",
        'paraphrases': [
            "What is the process of gradient descent?",
            "Can you explain the gradient descent algorithm?",
            "What are the steps in gradient descent optimization?"
        ]
    },
    {
        'original': "What is overfitting and how do we prevent it?",
        'paraphrases': [
            "How can we avoid overfitting in our models?",
            "What techniques help prevent a model from overfitting?",
            "When does overfitting occur and what are the solutions?"
        ]
    }
]

def compare_retrieval_consistency(question_set):
    original = question_set['original']
    paraphrases = question_set['paraphrases']
    
    # Get original results
    original_bm25 = set(doc for doc, _ in bm25_search(original))
    original_dense = set(doc for doc, _ in dense_search(original))
    
    # Compare with paraphrases
    bm25_consistency = []
    dense_consistency = []
    
    for paraphrase in paraphrases:
        para_bm25 = set(doc for doc, _ in bm25_search(paraphrase))
        para_dense = set(doc for doc, _ in dense_search(paraphrase))
        
        bm25_overlap = len(original_bm25.intersection(para_bm25)) / len(original_bm25)
        dense_overlap = len(original_dense.intersection(para_dense)) / len(original_dense)
        
        bm25_consistency.append(bm25_overlap)
        dense_consistency.append(dense_overlap)
    
    return {
        'bm25_avg_consistency': np.mean(bm25_consistency),
        'dense_avg_consistency': np.mean(dense_consistency)
    }

print("Analyzing retrieval consistency across paraphrases:")
for question_set in paraphrase_examples:
    print(f"\nOriginal question: {question_set['original']}")
    metrics = compare_retrieval_consistency(question_set)
    print(f"BM25 average consistency: {metrics['bm25_avg_consistency']:.2%}")
    print(f"Dense retrieval average consistency: {metrics['dense_avg_consistency']:.2%}")

## Discussion Points

1. Retrieval Method Comparison
   - When does BM25 perform better than dense retrieval?
   - How do the methods handle different types of semantic similarity?
   - What are the computational trade-offs?

2. Robustness Analysis
   - How well do the methods handle paraphrasing?
   - What types of questions are challenging for each method?
   - How could we improve robustness?

3. RAG System Design
   - How to choose the number of retrieved documents?
   - What's the impact of context window size?
   - How to handle document relevance in the prompt?

4. Practical Considerations
   - Scaling to larger knowledge bases
   - Handling document updates
   - Balancing accuracy and latency